In [1]:
# IMPORTS FROM PYOMO

from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    Objective,
    Expression,
    value,
    check_optimal_termination,
    assert_optimal_termination,
    TransformationFactory,
    units as pyunits,
)

from pyomo.util.check_units import assert_units_consistent
from pyomo.network import Arc



# IMPORTS FROM IDAES
from idaes.core import FlowsheetBlock, UnitModelCostingBlock
from idaes.models.unit_models import Feed, Product
from idaes.core.util.model_statistics import degrees_of_freedom
from idaes.core.util.scaling import calculate_scaling_factors, set_scaling_factor
from idaes.core.util.initialization import propagate_state



# IMPORTS FROM WaterTAP
from watertap.property_models.NaCl_prop_pack import NaClParameterBlock
from watertap.property_models.seawater_prop_pack import SeawaterParameterBlock
from watertap.unit_models.pressure_changer import Pump
from watertap.unit_models.reverse_osmosis_0D import (
    ReverseOsmosis0D,
    ConcentrationPolarizationType,
    MassTransferCoefficient,
    PressureChangeType,
)

from watertap.unit_models.zero_order import ChemicalAdditionZO
from watertap.unit_models.zero_order import UltraFiltrationZO
from watertap.unit_models.zero_order import ChlorinationZO
from watertap.unit_models.zero_order import DeepWellInjectionZO

from watertap.core.wt_database import Database                                                              # What is this line for?
from watertap.core.zero_order_properties import WaterParameterBlock as ZOProperties
from watertap.costing.zero_order_costing import ZeroOrderCosting
from watertap.costing import WaterTAPCosting
from watertap.core.solvers import get_solver


# TRANSLATOR FUNCTION

import idaes.logger as idaeslog
from idaes.core import declare_process_block_class
from idaes.core.util.exceptions import InitializationError
from idaes.models.unit_models.translator import TranslatorData

In [2]:
@declare_process_block_class("TranslatorZOtoSW")
class TranslatorZOtoSWData(TranslatorData):
    """
    Translator block for converting from ZO TDS to SW TDS
    """

    CONFIG = TranslatorData.CONFIG()

    def build(self):
        super().build()

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality mass flow water equation",
        )
        def eq_flow_mass_rule(blk, t):
            return (
                blk.properties_out[t].flow_mass_phase_comp["Liq", "H2O"]
                == blk.properties_in[t].flow_mass_comp["H2O"]
            )

        @self.Constraint(
            self.flowsheet().time,
            doc="Equality solute equation",
        )
        def eq_solute_mass_flow(blk, t):
            return (
                blk.properties_out[t].flow_mass_phase_comp["Liq", "NaCl"]
                == blk.properties_in[t].flow_mass_comp["NaCl"]
            )

        # ZO prop pack doesn't have temperature and pressure as state variables
        self.properties_out[0].pressure.fix(101325)
        self.properties_out[0].temperature.fix(298.15)

    def initialize_build(
        self,
        state_args_in=None,
        state_args_out=None,
        outlvl=idaeslog.NOTSET,
        solver=None,
        optarg=None,
    ):
        init_log = idaeslog.getInitLogger(self.name, outlvl, tag="unit")

        # Create solver
        opt = get_solver(solver, optarg)

        # ---------------------------------------------------------------------
        # Initialize state block
        flags = self.properties_in.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_in,
            hold_state=True,
        )

        self.properties_out.initialize(
            outlvl=outlvl,
            optarg=optarg,
            solver=solver,
            state_args=state_args_out,
        )

        if degrees_of_freedom(self) != 0:
            raise Exception(
                f"{self.name} degrees of freedom were not 0 at the beginning "
                f"of initialization. DoF = {degrees_of_freedom(self)}"
            )

        with idaeslog.solver_log(init_log, idaeslog.DEBUG) as slc:
            res = opt.solve(self, tee=slc.tee)

        self.properties_in.release_state(flags=flags, outlvl=outlvl)

        init_log.info(f"Initialization Complete: {idaeslog.condition(res)}")

        if not check_optimal_termination(res):
            raise InitializationError(
                f"{self.name} failed to initialize successfully. Please check "
                f"the output logs for more information."
            )


In [ ]:
# MODEL PARAMETERS

"""
solute_list = ["NaCl"]

Vol_flow_water = (10000000*3.785)/(1000*86400)* pyunits.m**3 / pyunits.s                        # Convert from Mgal/day to m3/s
Density  = 1000 * pyunits.kg / pyunits.m**3
mass_flow_water = Vol_flow_water*Density                                                        # Mass flow rate (Kg/s)
mass_flow_salt = 0.005                                                                          # Feed Concentration (Kg/m3)

use_default_removal = True

# Chemical dosing parameters
anti_scalant_dose = 1 * pyunits.mg / pyunits.liter

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 10 * pyunits.bar

# RO parameters
A_comp = 7.8994/(1000*3600*100000)      # membrane water permeability coeff [L/m2 * h * bar] to [m/Pa/s]
B_comp = 7e-11                          # membrane salt permeability coeff (m/s)
membrane_area = 90000  # m2
atmospheric = 101325  # Pa
deltaP = -3 * pyunits.bar
channel_height = (34*0.0254) * pyunits.mm       # 34 mil spacer height converted to mm
spacer_porosity = 0.75
RR = 0.95


print(mass_flow_water)
"""



solute_list = ["NaCl"]

mass_flow_water = 0.965 * pyunits.kg / pyunits.s
mass_flow_salt = 0.035 * pyunits.kg / pyunits.s

use_default_removal = True

# Chemical dosing parameters
anti_scalant_dose = 1 * pyunits.mg / pyunits.liter

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 75 * pyunits.bar

# RO parameters

A_comp =4.2e-12 * pyunits.m/(pyunits.s * pyunits.Pa)                    # membrane water permeability coefficient [m/s-Pa]
B_comp = 3e-8 * pyunits.m/(pyunits.s)                                   # membrane salt permeability coefficient [m/s]
membrane_area = 50  * pyunits.m**2
atmospheric = 101325  * pyunits.Pa
deltaP = -3 * pyunits.bar
channel_height = 1 * pyunits.mm 
spacer_porosity = 0.75  * pyunits.dimensionless
RR = 0.45   * pyunits.dimensionless

solver = get_solver()




In [4]:
m = ConcreteModel()
m.db = Database()                                                                                     # What is this line for?
m.fs = FlowsheetBlock(dynamic=False)


# Add property models
m.fs.zo_properties = ZOProperties(solute_list=solute_list)
m.fs.ro_properties = NaClParameterBlock()
#m.fs.ro_properties = SeawaterParameterBlock()

# Add unit models
m.fs.feed = Feed(property_package=m.fs.zo_properties)
# Set feed stream
m.fs.feed.properties[0].flow_mass_comp["H2O"].fix(mass_flow_water)
m.fs.feed.properties[0].flow_mass_comp["NaCl"].fix(mass_flow_salt)
#m.fs.feed.properties[0].flow_mass_comp["TDS"].fix(mass_flow_salt)


m.fs.ultra_filt = UltraFiltrationZO(property_package=m.fs.zo_properties, database=m.db)                                             # Where to see arguments required for each function
m.fs.chem_addition = ChemicalAdditionZO(property_package=m.fs.zo_properties,database=m.db,process_subtype="anti-scalant")           # Where to see arguments required for each function
m.fs.chem_addition.chemical_dosage.fix(anti_scalant_dose)
m.fs.translator_feed = TranslatorZOtoSW(inlet_property_package=m.fs.zo_properties,outlet_property_package=m.fs.ro_properties)       # How to fix this translation to NaCl instead of SW
#m.fs.translator_brine = TranslatorZOtoSW(inlet_property_package=m.fs.zo_properties,outlet_property_package=m.fs.ro_properties)




m.fs.pump = Pump(property_package=m.fs.ro_properties)
m.fs.pump.efficiency_pump.fix(pump_efficiency)
m.fs.pump.control_volume.properties_out[0].pressure.fix(operating_pressure)


m.fs.RO = ReverseOsmosis0D(
    property_package=m.fs.ro_properties,
    has_pressure_change=True,
    pressure_change_type=PressureChangeType.calculated,
    mass_transfer_coefficient=MassTransferCoefficient.calculated,
    concentration_polarization_type=ConcentrationPolarizationType.calculated,
    module_type="spiral_wound",
)

# Fix (2) membrane properties
m.fs.RO.A_comp.fix(A_comp)
m.fs.RO.B_comp.fix(B_comp)

# Fix (4) module specifications
m.fs.RO.feed_side.channel_height.fix(channel_height)
m.fs.RO.feed_side.spacer_porosity.fix(spacer_porosity)
m.fs.RO.area.fix(membrane_area)
m.fs.RO.deltaP.fix(deltaP)                                  #
#m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

# (1) outlet state variable
m.fs.RO.permeate.pressure[0].fix(atmospheric)



# Create an Expression to calculate flux in LMH
m.fs.RO.flux_LMH = Expression(
    expr=pyunits.convert(
        m.fs.RO.mixed_permeate[0].flow_vol_phase["Liq"] / m.fs.RO.area,
        to_units=pyunits.liter / (pyunits.m**2 * pyunits.hr),
    )
)

m.fs.product = Product(property_package=m.fs.ro_properties)
m.fs.product.properties[0].conc_mass_phase_comp                                                         # What is this line doing?

#m.fs.brine = Product(property_package=m.fs.ro_properties)
#m.fs.brine.properties[0].conc_mass_phase_comp                                                           # What is this line doing?

#m.fs.chlorination = ChlorinationZO(property_package=m.fs.zo_properties, database=m.db)                                             # Where to see arguments required for each function
#m.fs.dwi = DeepWellInjectionZO(property_package=m.fs.zo_properties,database=m.db)                                                  # Where to see arguments required for each function


# Connect unit models with Arcs

m.fs.feed_to_ultra = Arc(source=m.fs.feed.outlet, destination=m.fs.ultra_filt.inlet)
m.fs.ultra_to_chem = Arc(source=m.fs.ultra_filt.treated, destination=m.fs.chem_addition.inlet)
m.fs.chem_to_trans = Arc(source=m.fs.chem_addition.outlet, destination=m.fs.translator_feed.inlet)
m.fs.trans_to_pump = Arc(source=m.fs.translator_feed.outlet, destination=m.fs.pump.inlet)
m.fs.pump_to_RO = Arc(source=m.fs.pump.outlet, destination=m.fs.RO.inlet)
m.fs.RO_to_product = Arc(source=m.fs.RO.permeate, destination=m.fs.product.inlet)                       # How to add chlorination and remineralization after this?

#brine to dwi?
# product to chlorination?  - Permeate?


TransformationFactory("network.expand_arcs").apply_to(m)       

In [8]:
m.fs.feed.properties[0].flow_mass_comp.display()

flow_mass_comp : Mass flowrate of each component
    Size=2, Index=fs.zo_properties.component_list, Units=kg/s
    Key  : Lower : Value : Upper : Fixed : Stale : Domain
     H2O :     0 : 0.965 :  None :  True : False : PositiveReals
    NaCl :     0 : 0.035 :  None :  True : False : PositiveReals


In [181]:
# Set scaling factors
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e2, index=("Liq", "TDS"))

m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1, index=("H2O"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e2, index=("TDS"))

set_scaling_factor(m.fs.pump.control_volume.work, 1e-3)
set_scaling_factor(m.fs.RO.area, 1e-2)
calculate_scaling_factors(m)


# Release constraints related to low concentrations
for item in [m.fs.RO.permeate_side, m.fs.RO.feed_side.properties_interface]:
    for idx, param in item.items():
        param.molality_phase_comp["Liq", "NaCl"].setlb(1e-5)   ## None maybe...
        param.pressure_osm_phase["Liq"].setlb(10)

print(f"dof = {degrees_of_freedom(m)}")


2026-02-19 14:05:06 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.translator_feed.properties_out[0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:06 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.pump.control_volume.properties_in[0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:06 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.pump.control_volume.properties_out[0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.feed_side.properties_in[0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.feed_side.properties_out[0.0].flow_mass_phase_comp[Liq,NaCl]


2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.feed_side.properties_interface[0.0,0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.feed_side.properties_interface[0.0,1.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.permeate_side[0.0,0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.permeate_side[0.0,1.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.RO.mixed_permeate[0.0].flow_mass_phase_comp[Liq,NaCl]
2026-02-19 14:05:07 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.product.properties[0.0].flow_mass_phase_comp[Liq,NaCl]
dof = 5


In [182]:
# Add costing
#m.fs.ro_costing = WaterTAPCosting()                                                                       # What cost model should be used? Zero order or waterTap Costing?
m.fs.costing = ZeroOrderCosting()


#m.fs.costing = ZeroOrderCosting()
m.fs.costing.base_currency = pyunits.USD_2023


m.fs.ultra_filt.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
m.fs.chem_addition.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
m.fs.pump.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)
m.fs.RO.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.costing)

#m.fs.chlorination.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
#m.fs.dwi.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

m.fs.costing.cost_process()
m.fs.costing.add_LCOW(m.fs.product.properties[0].flow_vol_phase["Liq"])
m.fs.costing.add_specific_energy_consumption(m.fs.product.properties[0].flow_vol_phase["Liq"], name="SEC")




"""


# Add costing
m.fs.ro_costing = WaterTAPCosting()                                                                       # What cost model should be used? Zero order or waterTap Costing?
m.fs.zo_costing = ZeroOrderCosting()


#m.fs.costing = ZeroOrderCosting()
m.fs.ro_costing.base_currency = pyunits.USD_2023
m.fs.zo_costing.base_currency = pyunits.USD_2023


m.fs.ultra_filt.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
m.fs.chem_addition.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
m.fs.pump.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.ro_costing)
m.fs.RO.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.ro_costing)

#m.fs.chlorination.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)
#m.fs.dwi.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)

m.fs.costing.cost_process()
m.fs.costing.add_LCOW(m.fs.product.properties[0].flow_vol_phase["Liq"])
m.fs.costing.add_specific_energy_consumption(m.fs.product.properties[0].flow_vol_phase["Liq"], name="SEC")

"""


2026-02-19 14:05:07 [WARNING] idaes.core.base.costing_base: flow_expr is an expression with a lower bound of less than zero. Costing requires that all flows have a lower bound equal to or greater than zero to avoid negative costs.


'\n\n\n# Add costing\nm.fs.ro_costing = WaterTAPCosting()                                                                       # What cost model should be used? Zero order or waterTap Costing?\nm.fs.zo_costing = ZeroOrderCosting()\n\n\n#m.fs.costing = ZeroOrderCosting()\nm.fs.ro_costing.base_currency = pyunits.USD_2023\nm.fs.zo_costing.base_currency = pyunits.USD_2023\n\n\nm.fs.ultra_filt.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)\nm.fs.chem_addition.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)\nm.fs.pump.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.ro_costing)\nm.fs.RO.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.ro_costing)\n\n#m.fs.chlorination.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)\n#m.fs.dwi.costing = UnitModelCostingBlock(flowsheet_costing_block=m.fs.zo_costing)\n\nm.fs.costing.cost_process()\nm.fs.costing.add_LCOW(m.fs.product.properties[0].flow_vol_phase[

In [183]:
print(f"dof = {degrees_of_freedom(m)}")

dof = 5


In [184]:
solver = get_solver()
   
#solver.solve(m.fs.feed)

propagate_state(m.fs.feed_to_ultra)
m.fs.ultra_filt.load_parameters_from_database(use_default_removal=use_default_removal)              # What do I need this line?
m.fs.ultra_filt.initialize()

print(f"dof = {degrees_of_freedom(m)}")


2026-02-19 14:05:07 [INFO] idaes.init.fs.ultra_filt.properties_in: Initialization Complete.
2026-02-19 14:05:07 [INFO] idaes.init.fs.ultra_filt.properties_treated: Initialization Complete.
2026-02-19 14:05:07 [INFO] idaes.init.fs.ultra_filt.properties_byproduct: Initialization Complete.
2026-02-19 14:05:07 [INFO] idaes.init.fs.ultra_filt.properties_in: State Released.
2026-02-19 14:05:07 [INFO] idaes.init.fs.ultra_filt: Initialization Complete: optimal - Optimal Solution Found
dof = 2


In [185]:
propagate_state(m.fs.ultra_to_chem)
m.fs.chem_addition.load_parameters_from_database(use_default_removal=use_default_removal)
m.fs.chem_addition.initialize()

print(f"dof = {degrees_of_freedom(m)}")

2026-02-19 14:05:07 [INFO] idaes.init.fs.chem_addition.properties: Initialization Complete.
2026-02-19 14:05:07 [INFO] idaes.init.fs.chem_addition.properties: State Released.
2026-02-19 14:05:07 [INFO] idaes.init.fs.chem_addition: Initialization Complete: optimal - Optimal Solution Found
dof = 0


In [186]:
propagate_state(m.fs.chem_to_trans)
m.fs.translator_feed.initialize()

print(f"dof = {degrees_of_freedom(m)}")

2026-02-19 14:05:07 [INFO] idaes.init.fs.translator_feed.properties_in: Initialization Complete.
2026-02-19 14:05:07 [INFO] idaes.init.fs.translator_feed.properties_in: State Released.
2026-02-19 14:05:07 [INFO] idaes.init.fs.translator_feed: Initialization Complete: optimal - Optimal Solution Found
dof = 0


In [187]:
propagate_state(m.fs.trans_to_pump)
m.fs.pump.initialize()

print(f"dof = {degrees_of_freedom(m)}")

2026-02-19 14:05:08 [INFO] idaes.init.fs.pump.control_volume: Initialization Complete
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-02-19 14:05:08 [INFO] idaes.init.fs.pump: Initialization Complete: optimal - Optimal Solution Found
dof = 0


In [188]:
propagate_state(m.fs.pump_to_RO)

print(f"dof = {degrees_of_freedom(m)}")
m.fs.RO.initialize()



dof = 0
2026-02-19 14:05:08 [INFO] idaes.init.fs.RO.feed_side: Initialization Complete
2026-02-19 14:05:09 [INFO] idaes.init.fs.RO: Initialization Complete: optimal - Optimal Solution Found


In [189]:
propagate_state(m.fs.RO_to_product)
m.fs.product.initialize()

print(f"dof = {degrees_of_freedom(m)}")

2026-02-19 14:05:09 [INFO] idaes.init.fs.product: Initialization Complete.
dof = 0


In [190]:
assert degrees_of_freedom(m) == 0
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

In [191]:
m.fs.costing.LCOW.display()
m.fs.costing.SEC.display()
m.fs.costing.aggregate_flow_costs.display()
m.fs.costing.aggregate_flow_electricity.display()
m.fs.costing.SEC_component.display()

m.fs.costing.total_capital_cost.display()
m.fs.costing.total_operating_cost.display()
m.fs.pump.work_mechanical.display()
m.fs.RO.recovery_vol_phase.display()
m.fs.product.properties[0].conc_mass_phase_comp.display()

LCOW : Size=1
    Key  : Value
    None : 0.8694508238751422
SEC : Size=1
    Key  : Value
    None : 5.460589001743557
aggregate_flow_costs : Size=2, Index=fs.costing.used_flows, Units=USD_2023/a
    Key          : Lower : Value              : Upper : Fixed : Stale : Domain
    anti-scalant :  None : 1402.8420421977826 :  None : False : False :  Reals
     electricity :  None :  6119.186842404924 :  None : False : False :  Reals
aggregate_flow_electricity : Aggregate flow for electricity
    Size=1, Index=None, Units=kW
    Key  : Lower : Value             : Upper : Fixed : Stale : Domain
    None :  None : 8.932501529487753 :  None : False : False :  Reals
SEC_component : Size=3
    Key              : Value
    fs.chem_addition : 2.099836682922182e-06
             fs.pump :     4.951456631199703
       fs.ultra_filt :     0.509130270707172
total_capital_cost : Total capital cost of the process
    Size=1, Index=None, Units=USD_2023
    Key  : Lower : Value              : Upper : Fixe

In [192]:
#m.fs.pump.control_volume.properties_out[0].pressure.unfix()


In [193]:
#print(f"dof = {degrees_of_freedom(m)}")

In [194]:
#m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

In [195]:
assert degrees_of_freedom(m) == 0
solver = get_solver()
results = solver.solve(m)
assert_optimal_termination(results)

In [196]:
m.fs.RO.report()


Unit : fs.RO                                                               Time: 0.0
------------------------------------------------------------------------------------
    Unit Performance

    Variables: 

    Key                        : Value   : Units         : Fixed : Bounds
                 Membrane Area :  50.000 :    meter ** 2 :  True : (0.1, 100000.0)
    Solvent Mass Recovery Rate : 0.49315 : dimensionless : False : (0.01, 0.999999)
      Volumetric Recovery Rate : 0.48831 : dimensionless : False : (None, None)

------------------------------------------------------------------------------------
    Stream Table
                                               Units         Feed Inlet  Feed Outlet  Permeate Outlet
    flow_mass_phase_comp ('Liq', 'H2O')   kilogram / second     0.91675     0.46465        0.45210   
    flow_mass_phase_comp ('Liq', 'NaCl')  kilogram / second    0.035000    0.034907     9.3159e-05   
    temperature                                      kelvin 

In [ ]:

solute_list = ["NaCl"]

Vol_flow_water = 0.995                      # Convert from Mgal/day to m3/s
Density  = 1000 
mass_flow_water = Vol_flow_water                                                                # Mass flow rate (Kg/s)
mass_flow_salt = 0.0005                                                                         # Feed Concentration (Kg/m3)

use_default_removal = True

# Chemical dosing parameters
anti_scalant_dose = 1 * pyunits.mg / pyunits.liter

# Pump parameters
pump_efficiency = 0.85 * pyunits.dimensionless
operating_pressure = 10 * pyunits.bar

# RO parameters
A_comp = 7.8994/(1000*3600*100000)      # membrane water permeability coeff [L/m2 * h * bar] to [m/Pa/s]
B_comp = 7e-11                          # membrane salt permeability coeff (m/s)
membrane_area = 50 # m2
atmospheric = 101325  # Pa
deltaP = -3 * pyunits.bar
channel_height = (34*0.0254) * pyunits.mm       # 34 mil spacer height converted to mm
spacer_porosity = 0.75
RR = 0.95


print(mass_flow_water)


0.995


In [ ]:
m.fs.feed.properties[0].flow_mass_comp.display()

7.8994


In [ ]:


m.fs.feed.properties[0].flow_mass_comp["H2O"].fix(mass_flow_water)
m.fs.feed.properties[0].flow_mass_comp["NaCl"].fix(mass_flow_salt)


m.fs.chem_addition.chemical_dosage.fix(anti_scalant_dose)

m.fs.pump.efficiency_pump.fix(pump_efficiency)


m.fs.RO.A_comp.fix(A_comp)
m.fs.RO.B_comp.fix(B_comp)


m.fs.RO.feed_side.channel_height.fix(channel_height)
m.fs.RO.feed_side.spacer_porosity.fix(spacer_porosity)
m.fs.RO.area.fix(membrane_area)
m.fs.RO.deltaP.fix(deltaP)                                  #
#m.fs.RO.recovery_vol_phase[0.0, "Liq"].fix(RR)

# (1) outlet state variable
m.fs.RO.permeate.pressure[0].fix(atmospheric)

In [199]:
# Set scaling factors
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1, index=("Liq", "H2O"))
m.fs.ro_properties.set_default_scaling("flow_mass_phase_comp", 1e3, index=("Liq", "TDS"))

m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1, index=("H2O"))
m.fs.zo_properties.set_default_scaling("flow_mass_comp", 1e3, index=("TDS"))

set_scaling_factor(m.fs.pump.control_volume.work, 1e-3)
set_scaling_factor(m.fs.RO.area, 1e-2)
calculate_scaling_factors(m)


# Release constraints related to low concentrations
for item in [m.fs.RO.permeate_side, m.fs.RO.feed_side.properties_interface]:
    for idx, param in item.items():
        param.molality_phase_comp["Liq", "NaCl"].setlb(None)   ## None maybe...
        param.pressure_osm_phase["Liq"].setlb(None)

print(f"dof = {degrees_of_freedom(m)}")

dof = 0


In [200]:
assert degrees_of_freedom(m) == 0
solver = get_solver()
results = solver.solve(m,tee=True)
assert_optimal_termination(results)

ipopt-watertap: ipopt with user variable scaling and IDAES jacobian constraint scaling
Ipopt 3.13.2: tol=1e-08
constr_viol_tol=1e-08
acceptable_constr_viol_tol=1e-08
bound_relax_factor=0.0
honor_original_bounds=no
nlp_scaling_method=user-scaling


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, 

RuntimeError: Solver failed to return an optimal solution. Solution status: warning, Termination condition: infeasible

In [ ]:
from idaes.core.util.model_diagnostics import DiagnosticsToolbox
 
dt =DiagnosticsToolbox(m)

dt.report_structural_issues()

dt.report_numerical_issues()

Model Statistics

        Activated Blocks: 51 (Deactivated: 0)
        Free Variables in Activated Constraints: 181 (External: 0)
            Free Variables with only lower bounds: 69
            Free Variables with only upper bounds: 0
            Free Variables with upper and lower bounds: 95
        Fixed Variables in Activated Constraints: 54 (External: 0)
        Activated Equality Constraints: 181 (Deactivated: 0)
        Activated Inequality Constraints: 0 (Deactivated: 0)
        Activated Objectives: 0 (Deactivated: 0)

------------------------------------------------------------------------------------
1 WARNINGS


------------------------------------------------------------------------------------
2 Cautions

    Caution: 1 variable fixed to 0
    Caution: 22 unused variables (22 fixed)

------------------------------------------------------------------------------------
Suggested next steps:

    display_potential_evaluation_errors()

component keys that are not exported a

In [ ]:
dt.display_constraints_with_large_residuals()

The following constraint(s) have large residuals (>1.0E-05):

    fs.chem_addition.chemical_flow_constraint[0.0]: 1.93562E-04
    fs.chem_addition.costing.capital_cost_constraint: 1.36720E-04
    fs.RO.eq_flux_mass[0.0,0.0,Liq,NaCl]: 1.72025E-05
    fs.RO.eq_flux_mass[0.0,1.0,Liq,NaCl]: 2.91228E-05
    fs.RO.eq_connect_mass_transfer[0.0,Liq,NaCl]: 2.22728E-03
    fs.RO.eq_mass_frac_permeate[0.0,0.0,NaCl]: 1.71735E-05
    fs.RO.eq_mass_frac_permeate[0.0,1.0,NaCl]: 1.52352E-05
    fs.RO.feed_side.eq_N_Re[0.0,1.0]: 9.49491E-05



In [ ]:
dt.compute_infeasibility_explanation()


ERROR: Unable to clone Pyomo component attribute. Component 'FiniteSetOf'
contains an uncopyable field '_ref' (<class 'dict_keys'>).  Setting field to
`None` on new object
ERROR: Unable to clone Pyomo component attribute. Component 'FiniteSetOf'
contains an uncopyable field '_ref' (<class 'dict_keys'>).  Setting field to
`None` on new object
ERROR: Unable to clone Pyomo component attribute. Component 'FiniteSetOf'
contains an uncopyable field '_ref' (<class 'dict_keys'>).  Setting field to
`None` on new object
ERROR: Unable to clone Pyomo component attribute. Component 'FiniteSetOf'
contains an uncopyable field '_ref' (<class 'dict_keys'>).  Setting field to
`None` on new object


AttributeError: 'NoneType' object has no attribute 'set_value'

In [ ]:
dt.display_variables_at_or_outside_bounds()

The following variable(s) have values at or outside their bounds (tol=0.0E+00):

    fs.ro_properties.dens_mass_param['0'] (fixed): value=995 bounds=(995, 995)
    fs.ro_properties.dens_mass_param['1'] (fixed): value=756 bounds=(756, 756)
    fs.ro_properties.visc_d_param['0'] (fixed): value=0.00098 bounds=(0.00098, 0.00098)
    fs.ro_properties.visc_d_param['1'] (fixed): value=0.00215 bounds=(0.00215, 0.00215)
    fs.ro_properties.diffus_param['0'] (fixed): value=1.51e-09 bounds=(1.51e-09, 1.51e-09)
    fs.ro_properties.diffus_param['2'] (fixed): value=3.01e-08 bounds=(3.01e-08, 3.01e-08)
    fs.ro_properties.diffus_param['3'] (fixed): value=-1.22e-07 bounds=(-1.22e-07, -1.22e-07)
    fs.ro_properties.diffus_param['4'] (fixed): value=1.53e-07 bounds=(1.53e-07, 1.53e-07)
    fs.ro_properties.diffus_param['1'] (fixed): value=-2e-09 bounds=(-2e-09, -2e-09)
    fs.ro_properties.osm_coeff_param['0'] (fixed): value=0.918 bounds=(0.918, 0.918)
    fs.ro_properties.osm_coeff_param['1'] (fixed